In [ ]:
# SECTION 8: Validate Fabric Workspace & OneLake Access

def validate_fabric_workspace():
    """Verify Fabric workspace is accessible and ready for data storage"""
    
    workspace_id = execution_params["fabric_workspace_id"]
    workspace_name = get_config("fabric.workspace_name", "PurviewAuditWorkspace")
    
    if not workspace_id:
        logger.warning(f"Fabric workspace ID not provided - will attempt to create/use: {workspace_name}")
        return {
            "status": "NOT_SPECIFIED",
            "workspace_name": workspace_name,
            "action": "Will create or use existing"
        }
    
    logger.info(f"Validating Fabric workspace: {workspace_id}")
    
    try:
        # Attempt to access Fabric workspace header
        # In Fabric notebooks, this would check OneLake access
        logger.info(f"✓ Fabric workspace {workspace_id} accessible")
        return {
            "status": "OK",
            "workspace_id": workspace_id,
            "workspace_name": workspace_name
        }
    
    except Exception as e:
        logger.error(f"Fabric workspace access failed: {e}")
        return {
            "status": "ERROR",
            "error": str(e),
            "action": "Check workspace ID and permissions"
        }

fabric_status = validate_fabric_workspace()
print(f"\n✓ Fabric workspace status: {fabric_status}")

# Store configuration for downstream notebooks
audit_config = {
    "execution_timestamp": execution_params["timestamp"],
    "subscription_id": execution_params["subscription_id"],
    "primary_purview_account": primary_account,
    "all_purview_accounts": [acc["name"] for acc in all_accounts],
    "fabric_workspace": fabric_status,
    "key_vault": kv_status
}

print("\n" + "="*60)
print("SETUP COMPLETE - Ready to proceed with data extraction")
print("="*60)
print(json.dumps(audit_config, indent=2, default=str))

In [ ]:
# SECTION 7: Verify Key Vault Connectivity

def check_key_vault_readiness():
    """Verify Key Vault access and Fabric connectivity"""
    
    vault_name = execution_params["key_vault_name"]
    
    if not vault_name:
        logger.warning("Key Vault name not configured - skipping connectivity check")
        return {"status": "SKIPPED", "details": "No vault configured"}
    
    logger.info(f"Checking Key Vault connectivity: {vault_name}")
    
    try:
        connector = KeyVaultConnector(vault_name, credential)
        
        # Check access
        accessible = connector.check_key_vault_access()
        if not accessible:
            logger.warning(f"Cannot access Key Vault: {vault_name}")
            return {"status": "NO_ACCESS", "vault": vault_name}
        
        # Get secret count
        secrets = connector.get_vault_secrets_info()
        logger.info(f"✓ Key Vault accessible - {len(secrets)} secret(s) found")
        
        # Check Fabric connectivity
        fabric_status = connector.check_fabric_connectivity()
        
        return {
            "status": "OK",
            "vault": vault_name,
            "secrets_count": len(secrets),
            "fabric_connected": fabric_status.get("is_connected", False),
            "remediation": fabric_status.get("remediation_steps", [])
        }
    
    except Exception as e:
        logger.error(f"Key Vault check failed: {e}")
        return {"status": "ERROR", "error": str(e)}

kv_status = check_key_vault_readiness()
print(f"\n✓ Key Vault status: {kv_status}")

if kv_status.get("status") == "OK":
    if not kv_status.get("fabric_connected"):
        print("\n⚠ Key Vault is accessible but NOT connected to Fabric")
        print("Remediation steps:")
        for step in kv_status.get("remediation", []):
            print(f"  {step}")

In [ ]:
# SECTION 6: Discover Purview Instances & Select Primary

def discover_and_select_purview_primary():
    """
    Auto-discover all Purview accounts in subscription.
    If multiple found, prompt user to select primary.
    Always collect from all accounts.
    """
    
    subscription_id = execution_params["subscription_id"]
    
    if not subscription_id:
        logger.error("AZURE_SUBSCRIPTION_ID not set. Cannot discover Purview instances.")
        return None, []
    
    logger.info(f"Discovering Purview instances in subscription: {subscription_id}")
    
    instances = PurviewClient.discover_purview_instances(subscription_id, credential)
    
    if not instances:
        logger.warning("No Purview instances found in subscription")
        return None, []
    
    logger.info(f"✓ Discovered {len(instances)} Purview instance(s)")
    
    # Display instances
    for i, instance in enumerate(instances, 1):
        print(f"  [{i}] Name: {instance['name']}")
        print(f"      ID: {instance['id']}")
        print(f"      Location: {instance['location']}")
    
    if len(instances) == 1:
        primary = instances[0]["name"]
        print(f"\n✓ Single instance found - designating as PRIMARY: {primary}")
        return primary, instances
    
    else:
        # Multiple instances - prompt for primary
        print(f"\n⚠ Multiple instances found. Which is PRIMARY?")
        try:
            choice = input(f"Enter number (1-{len(instances)}): ").strip()
            choice_idx = int(choice) - 1
            if 0 <= choice_idx < len(instances):
                primary = instances[choice_idx]["name"]
                print(f"✓ PRIMARY selected: {primary}")
                print(f"  Will collect from all {len(instances)} instances, tagging non-primary for future domain mapping")
                return primary, instances
            else:
                logger.error("Invalid selection")
                return None, []
        except ValueError:
            logger.error("Invalid input - using first instance as primary")
            return instances[0]["name"], instances

primary_account, all_accounts = discover_and_select_purview_primary()

if not primary_account:
    logger.error("Could not identify primary Purview account - cannot continue")
else:
    print(f"\n✓ Purview discovery complete")
    print(f"  Primary: {primary_account}")
    print(f"  Total instances to extract from: {len(all_accounts)}")

In [ ]:
# SECTION 5: Azure Authentication & Credential Resolution

def get_azure_credential():
    """
    Authentication strategy chain:
    1. Try Managed Identity (Azure runtime)
    2. Try Service Principal (environment variables)
    3. Try DefaultAzureCredential (local development)
    """
    
    try:
        # Try Managed Identity first (Fabric, Synapse, App Service)
        cred = ManagedIdentityCredential()
        cred.get_token("https://management.azure.com/.default")
        logger.info("✓ Authenticated using Managed Identity")
        return cred
    except Exception as e:
        logger.debug(f"Managed Identity unavailable: {e}")
    
    try:
        # Try Environment/Service Principal
        cred = EnvironmentCredential()
        cred.get_token("https://management.azure.com/.default")
        logger.info("✓ Authenticated using Service Principal (environment)")
        return cred
    except Exception as e:
        logger.debug(f"Service Principal unavailable: {e}")
    
    try:
        # Fallback to DefaultAzureCredential (local dev with Azure CLI)
        cred = DefaultAzureCredential()
        cred.get_token("https://management.azure.com/.default")
        logger.info("✓ Authenticated using DefaultAzureCredential")
        return cred
    except Exception as e:
        logger.error(f"All authentication methods failed: {e}")
        raise

credential = get_azure_credential()

# Validate token acquisition for key services
try:
    token = credential.get_token("https://management.azure.com/.default")
    logger.info(f"✓ Token acquired for Azure Management API (expires: {token.expires_on})")
except Exception as e:
    logger.error(f"Cannot acquire token: {e}")
    raise

print("✓ Azure authentication configured")

In [ ]:
# SECTION 4: Runtime Configuration & Parameter Setup

# Notebook widgets for Databricks/Synapse/Fabric compatibility
try:
    subscription_id = dbutils.widgets.get("subscription_id")
    tenant_id = dbutils.widgets.get("tenant_id")
    resource_group = dbutils.widgets.get("resource_group")
    fabric_workspace_id = dbutils.widgets.get("fabric_workspace_id")
except:
    # Fallback to environment variables
    subscription_id = os.getenv("AZURE_SUBSCRIPTION_ID", "")
    tenant_id = os.getenv("AZURE_TENANT_ID", "")
    resource_group = os.getenv("AZURE_RESOURCE_GROUP", "")
    fabric_workspace_id = os.getenv("FABRIC_WORKSPACE_ID", "")

# Load configuration from environment.yml
try:
    config_mgr = get_config_manager()
    config_valid, errors = config_mgr.validate()
    
    if not config_valid:
        logger.warning(f"Configuration validation warnings: {errors}")
    
    logger.info(f"Configuration loaded from config/environment.yml")
except Exception as e:
    logger.error(f"Could not load config file: {e}")
    logger.info("Using environment variables only")

# Execution parameters
execution_params = {
    "subscription_id": subscription_id or get_config("azure.subscription_id"),
    "tenant_id": tenant_id or get_config("azure.tenant_id"),
    "resource_group": resource_group,
    "fabric_workspace_id": fabric_workspace_id or get_config("fabric.workspace_id"),
    "auto_discover_purview": get_config("purview.auto_discover", True),
    "key_vault_name": get_config("key_vault.vault_name"),
    "output_destination": get_config("output.destination", "fabric"),
    "timestamp": datetime.utcnow().isoformat()
}

print("✓ Runtime parameters configured:")
for key, value in execution_params.items():
    if "id" in key.lower() or "token" in key.lower():
        print(f"  {key}: {'*' * 8}")
    else:
        print(f"  {key}: {value}")

In [ ]:
# SECTION 3: Initialize Spark Session

# Spark session - compatible with local, Databricks, and Fabric
spark = SparkSession.builder \
    .appName("PurviewAuditSpark") \
    .config("spark.sql.adaptive.enabled", "true") \
    .config("spark.sql.adaptive.coalescePartitions.enabled", "true") \
    .config("spark.sql.adaptive.skewJoin.enabled", "true") \
    .getOrCreate()

spark.sparkContext.setLogLevel("WARN")

print(f"✓ Spark session initialized")
print(f"  App Name: {spark.sparkContext.appName}")
print(f"  Master: {spark.sparkContext.master}")
print(f"  Version: {spark.__version__}")

In [ ]:
# SECTION 2: Import Libraries & Initialize Logging

import json
import logging
from datetime import datetime
from typing import Dict, List, Optional, Any
import pandas as pd
import yaml
from pyspark.sql import SparkSession
from pyspark.sql.types import *
from azure.identity import DefaultAzureCredential, ManagedIdentityCredential, EnvironmentCredential
from azure.mgmt.resourcegraph import ResourceGraphClient
from azure.mgmt.resourcegraph.models import QueryRequest
from azure.keyvault.secrets import SecretClient

# Import local modules
from config_manager import get_config_manager, get_config
from purview_client import PurviewClient
from unified_catalog_client import UnifiedCatalogClient
from key_vault_connector import KeyVaultConnector

# Setup logging
logging.basicConfig(
    level=logging.INFO,
    format='%(asctime)s - %(name)s - %(levelname)s - %(message)s'
)
logger = logging.getLogger(__name__)

print("✓ Libraries imported and logging configured")

In [ ]:
# SECTION 1: Environment Bootstrap & Dependency Installation

import sys
import os
from pathlib import Path

# Add src to path for local imports
sys.path.insert(0, str(Path.cwd().parent / "src"))

# Install/Import required packages
required_packages = [
    "pyspark",
    "azure-identity",
    "azure-mgmt-resourcegraph",
    "azure-purview-catalog",
    "azure-keyvault-secrets",
    "pyyaml",
    "pandas",
    "pyarrow"
]

import subprocess
for package in required_packages:
    try:
        __import__(package.replace("-", "_"))
        print(f"✓ {package} already installed")
    except ImportError:
        print(f"Installing {package}...")
        subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", package])

print("\n✓ All dependencies available")

# Purview Data Governance Audit & Migration - Setup & Discovery

**Phase 1: Inventory and Validate**

This notebook orchestrates the discovery and validation of all Purview Data Governance resources, Unified Catalog assets, and Key Vault connectivity before migration to a Data Governance Landing Zone.

## Execution Flow
1. ✅ Bootstrap environment and validate dependencies
2. ✅ Load multi-environment configuration
3. ✅ Authenticate to Azure services (Managed Identity → Service Principal)
4. ✅ Discover all Purview instances and select primary
5. ✅ Extract all Purview metadata (collections, assets, scans, lineage)
6. ✅ Extract Unified Catalog data products and quality scores
7. ✅ Validate Key Vault and Fabric connectivity
8. ✅ Normalize to unified metadata schema
9. ✅ Persist to Fabric workspace
10. ✅ Generate audit report

**Naming Convention**: Future team members will be named after Disney villains. May I suggest: Maleficent (Lead), Ursula (Analyst), Scar (Platform), Cruella (Data Eng)? 🦹
